## Using DSPy to Generate Example Questions for Prompt Optimization

### Setting up DSPy

In [ ]:
# Import libraries
import random
import dspy
import json
from tqdm import tqdm

import os
from dotenv import load_dotenv
load_dotenv()
open_ai_api_key = os.getenv("OPENAI_API_KEY")


c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# create gpt-4 model
gpt4 = dspy.OpenAI(model='gpt-4-0125-preview', max_tokens=1100, api_key=open_ai_api_key)  
dspy.configure(lm=gpt4)

gpt4("which openai model are you? Are you gpt4?")

["I am based on OpenAI's technology, but I don't have the capability to update my own information or know about developments after my last update in 2023. As of my last update, I am designed based on the principles of models like GPT-3, but I can't claim to be GPT-4 or any specific version without more recent information. My responses are generated based on the knowledge and training I received up until my last update."]

In [3]:
# Create a signature

class CreateQuestions(dspy.Signature):
    """Create a variety of questions from a prompt. All of these questions will be asked to community members in an online survey to inform a decision around city resources and services. 
    The number of questions to generate is given as an input.
    Please generate questions that are different from the ones already generated, which are inputted.
    The output should be a list of JSONs with the following structure:
    [
        {
            "response_format": "open" or "closed",
            "description": string,
            "main_text": string,
            "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions
        }
    ]"""

    number = dspy.InputField(desc="An integer representing the number of questions to generate.")
    prompt = dspy.InputField(desc="The prompt to generate questions from.")
    prev_questions = dspy.InputField(desc="A list of questions that have already been generated. Each question is separated by a semi-colon.")
    # output_format = dspy.InputField(desc="The format of the output, which will be a JSON.")
    questions = dspy.OutputField(desc="The generated questions.")

### Helper functions

In [4]:
# helper function: check if a question was repeated
def is_unique_question(questions, new_question):
    for question in questions:
        question_details = question["question"]
        if question_details["main_text"] == new_question["main_text"]:
            return False
    return True

### [DROPPED] Generate questions for AssessApplicability

In [12]:
# helper function: generate questions based on alignment level
def generate_questions_by_score(question_generator, global_questions, repeated_questions, question_qualities, question_type, score, 
                                alignment_level, output_name, num_cycles=3, num_questions=1, total_qualities=5, qualities_sample=[]):
    """
    question_generator: the question generator model, 
    global_questions: list of JSONs with questions generated so far, 
    question_qualities: string with the qualities to include / exclude, 
    question_type: "open-ended" or "closed-ended", 
    score: "pass" or "fail",
    alignment_level: alignment level to generate questions for, 
    output_name: name of the output field in the JSON 
    num_cycles: number of cycles to generate questions, default is 3
    num_questions: number of questions to generate each cycle, default is 1 
    total_qualities: total number of qualities to sample from, default is 9
    qualities_sample: list of qualities to sample from or empty list if not applicable
    """
    question_type_outputs = []

    cycle_count = 0

    antecendent = ""

    # if alignment_level != "uphold all":
    #     antecendent = "Try to generate a range of questions that obviously and less obviously excludes particular qualities."
    
    while cycle_count < num_cycles:
        
        # generate the prompt
        if question_type == "open-ended":
            # model_input = {
            #     "number": str(num_questions),
            #     "prompt": f"Generate {question_type} questions that, for each question, {alignment_level} of the following {total_qualities} qualities:\n {question_qualities}\n {antecendent}",
            #     "prev_questions": repeated_questions,
            #     output_name: score
            # }
            model_input = {
                "number": str(num_questions),
                "prompt": f"Generate {question_type} questions that, for each question, {alignment_level}.",
                "prev_questions": repeated_questions,
                output_name: score
            }
        else:
            # model_input = {
            #     "number": str(num_questions),
            #     "prompt": f"Generate {question_type} questions that, for each question, {alignment_level} of the following {total_qualities} qualities:\n {question_qualities}\n {antecendent} Please generate questions with less than six response categories and only include or exclude qualities in the main text of the questions.",
            #     "prev_questions": repeated_questions,
            #     output_name: score
            # }
            model_input = {
                "number": str(num_questions),
                "prompt": f"Generate {question_type} questions that, for each question, {alignment_level}. Please generate questions with less than six response categories and only include or exclude qualities in the main text of the questions.",
                "prev_questions": repeated_questions,
                output_name: score
            }

        # print the prompt
        print(model_input["prompt"])

        # get the output
        rand_int = random.randint(1, 100)
        # print(rand_int)
        output = question_generator(number=model_input["number"], prompt=model_input["prompt"],
                                    prev_questions=model_input["prev_questions"], 
                                    config=dict(temperature=0.7+0.0001*rand_int))
        rationale = output.rationale
        print(f"Rationale: {rationale}")
        generated_questions = output.questions
        # print(generated_questions)
        # make sure it's a list of JSONs
        # convert string to list of JSONs
        try:
            generated_questions = json.loads(generated_questions)
            print(generated_questions)
            # store the question type in each json
            new_generated_questions = []
            have_repeat = False
            for question in generated_questions:
                # clear the "description" field in question (save it for now since it provides a rationale for the question)
                # question["description"] = ""
                if is_unique_question(global_questions+question_type_outputs, question):
                    new_generated_questions.append({output_name: model_input[output_name], "question": question, "alignment_level": alignment_level})
                else:
                    print(f"Repeated question: {question["main_text"]} at {question_type} {alignment_level}")
                    # add to repeated questions
                    if repeated_questions == "":
                        repeated_questions = question["main_text"]
                    else:
                        repeated_questions = repeated_questions + "; " + question["main_text"] 

                    have_repeat = True

            question_type_outputs = question_type_outputs + new_generated_questions

            if not have_repeat:
                cycle_count += 1
            
            # cycle_count += 1
            
        except Exception as e:
            print(f"Error with input: {model_input} at cycle {cycle_count}. Error is {e}")
            # print(output.questions)

    return question_type_outputs, repeated_questions

In [13]:
# let's generate the training and validation data to optimize AssessSpecificity

# create the training prompts

repeated_questions = ""

# # set repeated_questions to what is in repeated_questions.json
# with open('generated_questions/repeated_questions.json', 'r') as f:
#     repeated_questions = json.load(f)

print(repeated_questions, type(repeated_questions))

global_training_questions = []

# # set global_training_questions to what is in readability_outputs_train.json
# with open('generated_questions/specificity_outputs.json', 'r') as f:
#     global_training_questions = json.load(f)

print(len(global_training_questions))

# defining the predictor
question_generator = dspy.ChainOfThought(CreateQuestions)


# question_qualities = """
# (1) Question should be appropriate for a layperson.
# (2) Question should not delve into information and emotions that the respondent may be incapable of addressing because of social, psychological, or situational constraints."""

# question_qualities = """
# (1) Question should be appropriate for a layperson.
# (2) The chosen response format (open or closed) should be the most logical way of asking the question.
# (3) Question should not ask for personally identifiable information."""

output_name = "applicability"


score_configs = [
    # { 
    #     "score": "pass",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": "uphold all",
    #     "qualities_sample": []
    # },
    # { 
    #     "score": "fail",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": f"purposefully break quality {1} but uphold the rest",
    #     "qualities_sample": []
    # },
    # {
    #     "score": "fail",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": f"purposefully break quality {2} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { 
    #     "score": "fail",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": f"purposefully break quality {3} but uphold the rest",
    #     "qualities_sample": [],
    # },
    { 
        "score": "fail",
        "num_cycles": 1,
        "num_questions": 3,
        "alignment_level": f"delve into information and emotions that the respondent may be incapable of addressing because of social, psychological, or situational constraints",
        "qualities_sample": [],
    },
    # { 
    #     "score": "fail",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": f"purposefully break quality {4} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { 
    #     "score": "fail",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": f"uphold none",
    #     "qualities_sample": []
    # }
]

 <class 'str'>
0


In [14]:
with tqdm(total=2*len(score_configs)) as pbar:
    for score_config in tqdm(score_configs):
        for question_type in tqdm(["open-ended", "closed-ended"]):
            temp_questions, repeated_questions = generate_questions_by_score(question_generator, global_training_questions, repeated_questions,
                                                        question_qualities, question_type, score_config["score"], score_config["alignment_level"],
                                                        output_name, num_cycles=score_config["num_cycles"], 
                                                        num_questions=score_config["num_questions"], total_qualities=2)
            
            print(score_config["score"], question_type, len(temp_questions), score_config["num_cycles"]*score_config["num_questions"], score_config["alignment_level"])
            # print(repeated_questions)
            global_training_questions = global_training_questions + temp_questions

            # # save current global_training_questions as a JSON
            # with open('generated_questions/specificity_outputs.json', 'w') as f:
            #     json.dump(global_training_questions, f)
            
            pbar.update(1)


print(len(global_training_questions))

  0%|          | 0/2 [00:00<?, ?it/s]


Generate open-ended questions that, for each question, delve into information and emotions that the respondent may be incapable of addressing because of social, psychological, or situational constraints.


 50%|█████     | 1/2 [00:13<00:13, 13.27s/it]


Rationale: Reasoning: Let's think step by step in order to produce the questions. We need to create questions that are open-ended and encourage respondents to reflect on and share thoughts or feelings they might not typically express due to various constraints. Given there are no previous questions to avoid duplicating, we'll focus on crafting questions that tap into areas often constrained by societal norms, personal insecurities, or environmental factors. These questions should be designed in a way that they respect the respondent's sensitivity while encouraging introspection and honest sharing.
[{'response_format': 'open', 'description': 'Exploration of unvoiced community needs', 'main_text': "What are some needs or services you feel are lacking in our city that you've hesitated to voice, and why?", 'response_categories': []}, {'response_format': 'open', 'description': 'Understanding personal impact of city policies', 'main_text': 'Can you describe a situation where a city policy or

100%|██████████| 2/2 [00:43<00:00, 21.91s/it]

Rationale: Reasoning: Let's think step by step in order to produce the questions. We need to create closed-ended questions related to the prompt, ensuring they delve into potentially sensitive areas that respondents might find difficult to address openly. To do this, we should consider topics that are often subject to social, psychological, or situational constraints, such as financial stability, mental health, and personal safety. These areas are commonly sensitive and may elicit responses that are influenced by the mentioned constraints. Since no previous questions have been provided, we will start fresh and ensure our questions are crafted to be respectful yet insightful, with a focus on understanding the community's needs and perspectives without overstepping personal boundaries. We will create questions with response options that are inclusive, allowing for a range of experiences to be expressed.
[{'response_format': 'closed', 'description': 'This question aims to gauge feelings o

In [ ]:
# save repeated_questions as a JSON
# with open('generated_questions/repeated_questions.json', 'w') as f:
#     json.dump(repeated_questions, f)


### Generate questions for AssessSpecificity

In [6]:
# helper function: generate questions based on alignment level
def generate_questions_by_score(question_generator, global_questions, repeated_questions, question_qualities, question_type, score, 
                                alignment_level, output_name, num_cycles=3, num_questions=1, total_qualities=5, qualities_sample=[]):
    """
    question_generator: the question generator model, 
    global_questions: list of JSONs with questions generated so far, 
    question_qualities: string with the qualities to include / exclude, 
    question_type: "open-ended" or "closed-ended", 
    score: "pass" or "fail",
    alignment_level: alignment level to generate questions for, 
    output_name: name of the output field in the JSON 
    num_cycles: number of cycles to generate questions, default is 3
    num_questions: number of questions to generate each cycle, default is 1 
    total_qualities: total number of qualities to sample from, default is 9
    qualities_sample: list of qualities to sample from or empty list if not applicable
    """
    question_type_outputs = []

    cycle_count = 0

    antecendent = ""

    if alignment_level != "uphold all":
        antecendent = "Try to generate a range of questions that obviously and less obviously excludes particular qualities."
    
    while cycle_count < num_cycles:
        
        # generate the prompt
        if question_type == "open-ended":
            model_input = {
                "number": str(num_questions),
                "prompt": f"Generate {question_type} questions that, for each question, {alignment_level} of the following {total_qualities} qualities:\n {question_qualities}\n {antecendent}",
                "prev_questions": repeated_questions,
                output_name: score
            }
        else:
            model_input = {
                "number": str(num_questions),
                "prompt": f"Generate {question_type} questions that, for each question, {alignment_level} of the following {total_qualities} qualities:\n {question_qualities}\n {antecendent} Please generate questions with less than six response categories and only include or exclude qualities in the main text of the questions.",
                "prev_questions": repeated_questions,
                output_name: score
            }

        # print the prompt
        # print(model_input["prompt"])

        # get the output
        rand_int = random.randint(1, 100)
        # print(rand_int)
        output = question_generator(number=model_input["number"], prompt=model_input["prompt"],
                                    prev_questions=model_input["prev_questions"], 
                                    config=dict(temperature=0.7+0.0001*rand_int))
        rationale = output.rationale
        # print(f"Rationale: {rationale}")
        generated_questions = output.questions
        # print(generated_questions)
        # make sure it's a list of JSONs
        # convert string to list of JSONs
        try:
            generated_questions = json.loads(generated_questions)
            # print(generated_questions)
            # store the question type in each json
            new_generated_questions = []
            have_repeat = False
            for question in generated_questions:
                # clear the "description" field in question (save it for now since it provides a rationale for the question)
                # question["description"] = ""
                if is_unique_question(global_questions+question_type_outputs, question):
                    new_generated_questions.append({output_name: model_input[output_name], "question": question, "alignment_level": alignment_level})
                else:
                    print(f"Repeated question: {question["main_text"]} at {question_type} {alignment_level}")
                    # add to repeated questions
                    if repeated_questions == "":
                        repeated_questions = question["main_text"]
                    else:
                        repeated_questions = repeated_questions + "; " + question["main_text"] 

                    have_repeat = True

            question_type_outputs = question_type_outputs + new_generated_questions

            if not have_repeat:
                cycle_count += 1
            
            # cycle_count += 1
            
        except Exception as e:
            print(f"Error with input: {model_input} at cycle {cycle_count}. Error is {e}")
            # print(output.questions)

    return question_type_outputs, repeated_questions

In [12]:
# let's generate the training and validation data to optimize AssessSpecificity

# create the training prompts

# repeated_questions = ""

# set repeated_questions to what is in repeated_questions.json
with open('generated_questions/repeated_questions.json', 'r') as f:
    repeated_questions = json.load(f)

print(repeated_questions, type(repeated_questions))

# global_training_questions = []

# set global_training_questions to what is in readability_outputs_train.json
with open('generated_questions/specificity_outputs.json', 'r') as f:
    global_training_questions = json.load(f)

print(len(global_training_questions))

# defining the predictor
question_generator = dspy.ChainOfThought(CreateQuestions)

# question_qualities = """
# (1) Question should measure only one underlying concept.
# (2) Question should refer to a specific reference frame (e.g. times and places) that is clear to the respondent.
# (3) Question should not contain any ambiguous words that could be interpreted in multiple ways. Question should contain terms that will have the same specific meaning to all respondents.
# (4) Question, not including the response categories, should not contain any predicates whose meanings are relative rather than absolute, as it is the case with quantitative adjectives or adverbs (e.g. often and frequently)."""

question_qualities = """
(1) Question should measure only one underlying concept.
(2) Question should refer to a specific reference frame (e.g. times and places) that is clear to the respondent.
(3) Question should not contain any ambiguous words that could be interpreted in multiple ways. Question should contain terms that will have the same specific meaning to all respondents."""

output_name = "specificity"


score_configs = [
    { 
        "score": "pass",
        "num_cycles": 8,
        "num_questions": 6,
        "alignment_level": "uphold all",
        "qualities_sample": []
    },
    # { # DONE
    #     "score": "fail",
    #     "num_cycles": 2,
    #     "num_questions": 6,
    #     "alignment_level": f"purposefully break quality {1} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "fail",
    #     "num_cycles": 2,
    #     "num_questions": 6,
    #     "alignment_level": f"purposefully break quality {2} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "fail",
    #     "num_cycles": 2,
    #     "num_questions": 6,
    #     "alignment_level": f"purposefully break quality {3} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DROPPED
    #     "score": "fail",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": f"purposefully break quality {4} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "fail",
    #     "num_cycles": 2,
    #     "num_questions": 6,
    #     "alignment_level": f"uphold none",
    #     "qualities_sample": []
    # }
]

How satisfied are you with the city's public transportation system?; How often do you use city parks?; Do you feel safe walking in your neighborhood?; Are the city's waste management services effective?; Which type of city project do you think should be prioritized?; Do you agree with the current city policies? <class 'str'>
96


In [13]:
with tqdm(total=2*len(score_configs)) as pbar:
    for score_config in tqdm(score_configs):
        for question_type in tqdm(["open-ended", "closed-ended"]):
            temp_questions, repeated_questions = generate_questions_by_score(question_generator, global_training_questions, repeated_questions,
                                                        question_qualities, question_type, score_config["score"], score_config["alignment_level"],
                                                        output_name, num_cycles=score_config["num_cycles"], 
                                                        num_questions=score_config["num_questions"], total_qualities=3)
            
            print(score_config["score"], question_type, len(temp_questions), score_config["num_cycles"]*score_config["num_questions"], score_config["alignment_level"])
            # print(repeated_questions)
            global_training_questions = global_training_questions + temp_questions

            # save current global_training_questions as a JSON
            with open('generated_questions/specificity_outputs.json', 'w') as f:
                json.dump(global_training_questions, f)
            
            pbar.update(1)


print(len(global_training_questions))

  0%|          | 0/2 [00:00<?, ?it/s]


Error with input: {'number': '6', 'prompt': 'Generate open-ended questions that, for each question, uphold all of the following 3 qualities:\n \n(1) Question should measure only one underlying concept.\n(2) Question should refer to a specific reference frame (e.g. times and places) that is clear to the respondent.\n(3) Question should not contain any ambiguous words that could be interpreted in multiple ways. Question should contain terms that will have the same specific meaning to all respondents.\n ', 'prev_questions': "How satisfied are you with the city's public transportation system?; How often do you use city parks?; Do you feel safe walking in your neighborhood?; Are the city's waste management services effective?; Which type of city project do you think should be prioritized?; Do you agree with the current city policies?", 'specificity': 'pass'} at cycle 3. Error is Extra data: line 1 column 2 (char 1)


 50%|█████     | 1/2 [03:16<03:16, 196.73s/it]


pass open-ended 48 48 uphold all
Repeated question: Do you find the street lighting in your area to be adequate for nighttime visibility? at closed-ended uphold all
Repeated question: How would you rate the quality of tap water in your home? at closed-ended uphold all
Repeated question: How satisfied are you with the response time of emergency services in your area? at closed-ended uphold all


100%|██████████| 2/2 [10:36<00:00, 318.10s/it]

pass closed-ended 63 48 uphold all
207


In [14]:
# save repeated_questions as a JSON
# with open('generated_questions/repeated_questions.json', 'w') as f:
#     json.dump(repeated_questions, f)


### Generate questions for AssessBias

In [62]:
# helper function: generate questions based on bias type
def generate_questions_by_bias_type(question_generator, global_questions, repeated_questions, 
                                    question_type, config, alignment_level):
    """
    question_generator: the question generator model, 
    global_questions: list of JSONs with questions generated so far, 
    repeated_questions: list of questions that have been repeated to put in the prompt,
    question_type: "open-ended" or "closed-ended", 
    config: a dictionary with the following keys:
    { 
        "bias"
        "bias_name"
        "definition"
        "qualities"
        "anti_qualities"
        "num_cycles"
        "num_questions"
    },
    alignment_level: "contains" or "does not contain"

    """
    question_type_outputs = []

    cycle_count = 0
    
    while cycle_count < config["num_cycles"]:

        # generate the first part of the prompt based on alignment_level
        prompt_p1 = ""
        bias_value = ""
        if alignment_level == "contains":
            prompt_p1 = f"Generate {question_type} questions that, for each question, {alignment_level} the following bias: {config["bias_name"]}. {config["definition"]} {config["qualities"]} Try to generate a range of obviously biased and less obviously biased questions."
            bias_value = f"true"
        else:
            prompt_p1 = f"Generate {question_type} questions that, for each question, {alignment_level} the following bias: {config["bias_name"]}. {config["definition"]} {config["anti_qualities"]}"
            bias_value = f"false"

        # generate the prompt
        if question_type == "open-ended":
            model_input = {
                "number": str(config["num_questions"]),
                "prompt": prompt_p1,
                "prev_questions": repeated_questions,
                "bias": bias_value,
            }
        else:
            model_input = {
                "number": str(config["num_questions"]),
                "prompt": f"{prompt_p1} Please generate questions with less than six response categories.",
                "prev_questions": repeated_questions,
                "bias": bias_value
            }

        # print the prompt
        # print(model_input["prompt"])

        # get the output
        rand_int = random.randint(1, 100)
        # print(rand_int)
        output = question_generator(number=model_input["number"], prompt=model_input["prompt"],
                                    prev_questions=model_input["prev_questions"], 
                                    config=dict(temperature=0.7+0.0001*rand_int))
        rationale = output.rationale
        # print(f"Rationale: {rationale}")
        generated_questions = output.questions
        # print(generated_questions)
        # make sure it's a list of JSONs
        # convert string to list of JSONs
        try:
            generated_questions = json.loads(generated_questions)
            # print(generated_questions)
            # store the question type in each json
            new_generated_questions = []
            have_repeat = False
            for question in generated_questions:
                # clear the "description" field in question (save it for now since it provides a rationale for the question)
                # question["description"] = ""
                if is_unique_question(global_questions+question_type_outputs, question):
                    new_generated_questions.append({"bias": model_input["bias"], "question": question, "bias_type": config["bias"]})
                else:
                    print(f"Repeated question: {question["main_text"]} at {question_type} {alignment_level} {config["bias"]}")
                    # add to repeated questions
                    if repeated_questions == "":
                        repeated_questions = question["main_text"]
                    else:
                        repeated_questions = repeated_questions + "; " + question["main_text"] 

                    have_repeat = True

            question_type_outputs = question_type_outputs + new_generated_questions

            if not have_repeat:
                cycle_count += 1
            
            # cycle_count += 1
            
        except Exception as e:
            print(f"Error with input: {model_input} at cycle {cycle_count}. Error is {e}")
            # print(output.questions)

    return question_type_outputs, repeated_questions

In [70]:

# repeated_questions = ""

# set repeated_questions to what is in repeated_questions.json
with open('generated_questions/repeated_questions.json', 'r') as f:
    repeated_questions = json.load(f)

print(repeated_questions, type(repeated_questions))

# global_questions = []

# set global_questions to what is in readability_outputs_train.json
with open('generated_questions/bias_outputs.json', 'r') as f:
    global_questions = json.load(f)

print(len(global_questions))

# defining the predictor
question_generator = dspy.ChainOfThought(CreateQuestions)

output_name = "bias"

How satisfied are you with the city's public transportation system?; Why do you think the city has been neglecting the maintenance of public parks you frequently visit?; How has the city's failure to implement effective environmental policies affected your daily life?; What are the key areas where you believe the local police department has failed in ensuring public safety?; In what ways have the city’s recent zoning changes disadvantaged you or your community?; What improvements do you think are necessary for the public transportation options you reluctantly use?; How has the apparent underfunding of the city's education system impacted students, in your opinion?; How often do you visit city parks?; How safe do you feel in your community?; How adequate do you believe the funding for the city's education system is?; How satisfied are you with the city's waste management services?; How often do you use the city's public transportation system?; In what ways do you think the city could be

In [71]:
prompt_configs = [
    # { # done
    #     "bias": "leading_question",
    #     "bias_name": "leading question",
    #     "definition": "Leading questions are phrased in a way that suggests a particular answer is more desirable or correct.",
    #     "qualities": "They tend to have subjective adjectives, or context-laden words that frame the question in a positive or negative light.",
    #     "anti_qualities": "To avoid this bias, you could phrase your questions objectively, and provide answer scales with equally balanced negative and positive options. You could also include a counter-biasing statement to signal neutrality or avoid giving reasons for a given behavior in your question.", 
    #     "num_cycles": 1,
    #     "num_questions": 3,
    # },
    # {  # done
    #     "bias": "assumptions",
    #     "bias_name": "making assumptions about respondents",
    #     "definition": "These questions make assumptions on people’s behaviors, attitudes, or goals.",
    #     "qualities": "They tend to guess information instead of asking for it.",
    #     "anti_qualities": "", 
    #     "num_cycles": 1,
    #     "num_questions": 3,
    # },
    # { # done
    #     "bias": "double_barreled",
    #     "bias_name": "double-barreled questions",
    #     "definition": "Double-barreled questions ask about two or more things simultaneously.",
    #     "qualities": "",
    #     "anti_qualities": "", 
    #     "num_cycles": 4,
    #     "num_questions": 6,
    # },
    # { 
    #     "bias": "emotional_language",
    #     "bias_name": "emotionally loaded language",
    #     "definition": "These questions use emotionally loaded terms or phrases that imply judgment or assume a particular stance.",
    #     "qualities": "Words that carry strong positive or negative connotations can influence respondents' emotions and responses.",
    #     "anti_qualities": "", 
    #     "num_cycles": 4,
    #     "num_questions": 6,
    # },
]

In [72]:
with tqdm(total=2*2*len(prompt_configs)) as pbar:
    for config in tqdm(prompt_configs):
        for question_type in tqdm(["open-ended", "closed-ended"]):
            for alignment_level in ["contains", "does not contain"]:
                temp_questions, repeated_questions = generate_questions_by_bias_type(question_generator, global_questions, repeated_questions, 
                                        question_type, config, alignment_level)
                
                print(alignment_level, config["bias_name"], question_type, len(temp_questions), config["num_cycles"]*config["num_questions"])
                # print(repeated_questions)
                global_questions = global_questions + temp_questions

                # save current global_questions as a JSON
                with open('generated_questions/bias_outputs.json', 'w') as f:
                    json.dump(global_questions, f)
                
                pbar.update(1)


print(len(global_questions))

 25%|██▌       | 1/4 [00:17<00:51, 17.03s/it]

contains making assumptions about respondents open-ended 3 3
384


In [45]:
# save repeated_questions as a JSON
with open('generated_questions/repeated_questions.json', 'w') as f:
    json.dump(repeated_questions, f)


### Generate questions for AssessReadability

In [ ]:
# helper function: generate questions based on alignment level
def generate_questions_by_score(question_generator, global_questions, repeated_questions, question_qualities, question_type, score, 
                                alignment_level, output_name, num_cycles=3, num_questions=1, total_qualities=9, qualities_sample=[]):
    """
    question_generator: the question generator model, 
    global_questions: list of JSONs with questions generated so far, 
    question_qualities: string with the qualities to include / exclude, 
    question_type: "open-ended" or "closed-ended", 
    score: "low", "medium", or "high",
    alignment_level: alignment level to generate questions for, 
    output_name: name of the output field in the JSON 
    num_cycles: number of cycles to generate questions, default is 3
    num_questions: number of questions to generate each cycle, default is 1 
    total_qualities: total number of qualities to sample from, default is 9
    qualities_sample: list of qualities to sample from or empty list if not applicable
    """
    question_type_outputs = []

    cycle_count = 0
    
    while cycle_count < num_cycles:

        # NOTE: this method is no longer used (now it's inputted)
        # # get the alignment_level
        # alignment_level= ""
        # qualities_str = ""
        # if score == 1.0:
        #     alignment_level = "uphold all"
        #     qualities_str = "all"
        # elif score == 0.0:
        #     alignment_level = "uphold none"
        #     qualities_str = "none"
        # else:
        #     num_qualities = 10 - int(score*10)
        #     qualities_str = generate_qualities_to_exclude(num_qualities, total_qualities=total_qualities, qualities_sample=qualities_sample)
        #     alignment_level = f"do not uphold qualities {qualities_str} but uphold the rest"

        # generate the prompt
        if question_type == "open-ended":
            model_input = {
                "number": str(num_questions),
                "prompt": f"Generate {question_type} questions that, for each question, {alignment_level} of the following {total_qualities} qualities:\n {question_qualities}.",
                "prev_questions": repeated_questions,
                output_name: score
            }
        else:
            model_input = {
                "number": str(num_questions),
                "prompt": f"Generate {question_type} questions that, for each question, {alignment_level} of the following {total_qualities} qualities:\n {question_qualities}. Please generate questions with less than six response categories and only include or exclude qualities in the main text of the questions.",
                "prev_questions": repeated_questions,
                output_name: score
            }

        # print the prompt
        # print(model_input["prompt"])

        # get the output
        rand_int = random.randint(1, 100)
        # print(rand_int)
        output = question_generator(number=model_input["number"], prompt=model_input["prompt"],
                                    prev_questions=model_input["prev_questions"], 
                                    config=dict(temperature=0.7+0.0001*rand_int))
        rationale = output.rationale
        # print(f"Rationale: {rationale}")
        generated_questions = output.questions
        # print(generated_questions)
        # make sure it's a list of JSONs
        # convert string to list of JSONs
        try:
            generated_questions = json.loads(generated_questions)
            # print(generated_questions)
            # store the question type in each json
            new_generated_questions = []
            have_repeat = False
            for question in generated_questions:
                # clear the "description" field in question (save it for now since it provides a rationale for the question)
                # question["description"] = ""
                if is_unique_question(global_questions+question_type_outputs, question):
                    new_generated_questions.append({output_name: model_input[output_name], "question": question, "alignment_level": alignment_level})
                else:
                    print(f"Repeated question: {question["main_text"]} at {question_type} {alignment_level}")
                    # add to repeated questions
                    if repeated_questions == "":
                        repeated_questions = question["main_text"]
                    else:
                        repeated_questions = repeated_questions + "; " + question["main_text"] 

                    have_repeat = True

            question_type_outputs = question_type_outputs + new_generated_questions

            if not have_repeat:
                cycle_count += 1
            
            # cycle_count += 1
            
        except Exception as e:
            print(f"Error with input: {model_input} at cycle {cycle_count}. Error is {e}")
            # print(output.questions)

    return question_type_outputs, repeated_questions

In [ ]:
# let's generate the training and validation data to optimize AssessReadability

# create the training prompts

# repeated_questions = ""

# set repeated_questions to what is in repeated_questions.json
with open('generated_questions/repeated_questions.json', 'r') as f:
    repeated_questions = json.load(f)

# print(repeated_questions, type(repeated_questions))

# global_training_questions = []

# # set global_training_questions to what is in readability_outputs_train.json
with open('generated_questions/readability_outputs3.json', 'r') as f:
    global_training_questions = json.load(f)

print(len(global_training_questions))

# defining the predictor
question_generator = dspy.ChainOfThought(CreateQuestions)

# full list
question_qualities = """
(1) Question should meet a third-grade reading level.
(2) Question should not contain basic spelling or grammar mistakes.
(3) Question should be concise.
(4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or group and are difficult for others to understand) that are not defined.
(5) Question should not contain any acronyms that are not defined.
(6) Question should not mention proper nouns (e.g. names of specific people, places, or organizations) without describing what they are.
(7) Question should be in active voice.
(8) Question should have as few propositions and logical operators as possible.
(9) Question should not have negatives or double negatives."""

output_name = "readability"

# I do these one at a time to be safe
# NOTE: not longer used
# score_configs = [
#     {"score": 1.0, "num_cycles": 5, "num_questions": 2, "qualities_sample": []}, # done
#     {"score": 0.9, "num_cycles": 13, "num_questions": 1, "qualities_sample": [7, 8, 9]}, # done
#     {"score": 0.8, "num_cycles": 13, "num_questions": 1, "qualities_sample": [7, 8, 9]}, # done
#     {"score": 0.7, "num_cycles": 13, "num_questions": 1, "qualities_sample": [3, 4, 5, 6, 7, 8, 10]}, # done
#     {"score": 0.3, "num_cycles": 10, "num_questions": 1, "qualities_sample": []}, # done
#     {"score": 0.2, "num_cycles": 10, "num_questions": 1, "qualities_sample": []}, # done
#     {"score": 0.1, "num_cycles": 10, "num_questions": 1, "qualities_sample": []}, # done
#     {"score": 0.0, "num_cycles": 5, "num_questions": 2, "qualities_sample": []}, # done
# ]

score_configs = [
    # { # DONE
    #     "score": "high",
    #     "num_cycles": 5,
    #     "num_questions": 6,
    #     "alignment_level": "uphold all",
    #     "qualities_sample": []
    # },
    # { # not super consistent # DONE
    #     "score": "medium",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": f"purposefully break quality {6} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "medium",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": f"purposefully break quality {7} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "medium",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": f"purposefully break quality {8} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "medium",
    #     "num_cycles": 1,
    #     "num_questions": 3,
    #     "alignment_level": f"purposefully break quality {9} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 2,
    #     "alignment_level": f"uphold none",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 4,
    #     "alignment_level": f"purposefully break quality {1} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 4,
    #     "alignment_level": f"purposefully break quality {2} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 4,
    #     "alignment_level": f"purposefully break quality {3} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 4,
    #     "alignment_level": f"purposefully break quality {4} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 2,
    #     "alignment_level": f"purposefully break quality {5} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DROPPED (because of problem with proper nouns)
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 2,
    #     "alignment_level": f"purposefully break qualities {6} and {7} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DROPPED (because of problem with proper nouns)
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 2,
    #     "alignment_level": f"purposefully break qualities {6} and {8} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DROPPED (because of problem with proper nouns)
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 2,
    #     "alignment_level": f"purposefully break qualities {6} and {9} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 5,
    #     "alignment_level": f"purposefully break qualities {7} and {8} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 5,
    #     "alignment_level": f"purposefully break qualities {7} and {9} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 5,
    #     "alignment_level": f"purposefully break qualities {8} and {9} but uphold the rest",
    #     "qualities_sample": []
    # },
    # { # DONE
    #     "score": "low",
    #     "num_cycles": 1,
    #     "num_questions": 5,
    #     "alignment_level": f"purposefully break qualities {7}, {8}, and {9} but uphold the rest",
    #     "qualities_sample": []
    # },
]

In [ ]:
with tqdm(total=2*len(score_configs)) as pbar:
    for score_config in tqdm(score_configs):
        for question_type in tqdm(["open-ended", "closed-ended"]):
            temp_questions, repeated_questions = generate_questions_by_score(question_generator, global_training_questions, repeated_questions,
                                                        question_qualities, question_type, score_config["score"], score_config["alignment_level"],
                                                        output_name, num_cycles=score_config["num_cycles"], 
                                                        num_questions=score_config["num_questions"])
            
            print(score_config["score"], question_type, len(temp_questions), score_config["num_cycles"]*score_config["num_questions"], score_config["alignment_level"])
            # print(repeated_questions)
            global_training_questions = global_training_questions + temp_questions

            # # save current global_training_questions as a JSON
            with open('generated_questions/readability_outputs3.json', 'w') as f:
                json.dump(global_training_questions, f)
            
            pbar.update(1)


print(len(global_training_questions))

In [ ]:
# save repeated_questions as a JSON
# with open('generated_questions/repeated_questions.json', 'w') as f:
#     json.dump(repeated_questions, f)


### Post-processing of questions

In [20]:
# post-processing of questions

filename = "specificity_outputs"

# load the data 
with open(f"generated_questions/{filename}.json", 'r') as f:
    generated_questions = json.load(f)

print(len(generated_questions))

192


In [21]:
# randomly split each question_type and alignment_level into training and validation with 80% training and 20% validation
threshold = 0.75
training_questions = []
validation_questions = []

# put index of each question in a dictionary with key as the score and question type
# randomly shuffle the indices
# split the indices into training and validation
# get the questions based on the indices
# save the questions as a JSON

# get the indices
indices = {}
for i, question in enumerate(generated_questions):
    specificity = question["specificity"]
    question_type = question["question"]["response_format"]
    alignment_level = question["alignment_level"]
    key = f"{specificity}_{alignment_level}_{question_type}"
    if key not in indices:
        indices[key] = []
    indices[key].append(i)

# print(indices)

# print length of each key
for key in indices:
    print(key, len(indices[key]))

fail_purposefully break quality 1 but uphold the rest_open 12
fail_purposefully break quality 1 but uphold the rest_closed 12
fail_purposefully break quality 2 but uphold the rest_open 12
fail_purposefully break quality 2 but uphold the rest_closed 12
fail_purposefully break quality 3 but uphold the rest_open 12
fail_purposefully break quality 3 but uphold the rest_closed 12
fail_uphold none_open 12
fail_uphold none_closed 12
pass_uphold all_open 48
pass_uphold all_closed 48


In [22]:
# shuffle the indices
for key in indices:
    random.shuffle(indices[key])

# print(indices)

# split the indices
for key in indices:
    split = int(len(indices[key])*threshold)
    # print(len(indices[key]), split)
    training_indices = indices[key][:split]
    validation_indices = indices[key][split:]
    for idx in training_indices:
        training_questions.append(generated_questions[idx])
    for idx in validation_indices:
        validation_questions.append(generated_questions[idx])

print(len(training_questions), len(validation_questions))

# save the training and validation questions as JSONs
with open(f"generated_questions/{filename}_train.json", 'w') as f:
    json.dump(training_questions, f)

with open(f"generated_questions/{filename}_val.json", 'w') as f:
    json.dump(validation_questions, f)

144 48


In [23]:
# shuffle the order of the questions in generated_questions/readability_outputs_train.json
with open(f"generated_questions/{filename}_train.json", 'r') as f:
    generated_questions_train = json.load(f)

random.shuffle(generated_questions_train)

with open(f"generated_questions/{filename}_train.json", 'w') as f:
    json.dump(generated_questions_train, f)

# shuffle the order of the questions in generated_questions/readability_outputs_train.json
with open(f"generated_questions/{filename}_val.json", 'r') as f:
    generated_questions_val = json.load(f)

random.shuffle(generated_questions_val)

with open(f"generated_questions/{filename}_val.json", 'w') as f:
    json.dump(generated_questions_val, f)